# Vietnamese Economic News QA — Training & Evaluation

**Pipeline:** RAG (BGE-M3 Hybrid Retrieval + Cross-Encoder Reranker) → Fine-tune Generator (Qwen3-0.6B)  
**Evaluation:** ROUGE-1/2/L + BERTScore-F1 + RAGAS + Factual Correctness (Gemini 2.5 Pro)  
**QA Types:** FACTOID · SUMMARY · VERIFICATION · COMPARISON

| Section | Nội dung |
|---------|----------|
| 1 | Cài đặt & Import |
| 2 | Cấu hình tập trung |
| 3 | Load & kiểm tra dataset |
| 4 | Xây dựng Vector Store (Offline RAG) |
| 5 | Hybrid Retrieval + Reranker |
| 6 | Fine-tune Generator |
| 7 | Inference (sinh câu trả lời) |
| 8 | Evaluation — ROUGE + BERTScore + RAGAS |
| 9 | Kết quả & Visualisation |


## 1. Cài đặt & Import

In [ ]:
%%capture
%pip install pandas matplotlib numpy transformers torch bitsandbytes
%pip install accelerate sentence-transformers faiss-cpu rank_bm25 rouge-score bert-score sentencepiece tqdm
%pip install ragas datasets langchain-google-genai

In [ ]:
import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    GenerationConfig, EarlyStoppingCallback,
    )
from transformers.trainer_utils import get_last_checkpoint
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 2. Cấu hình tập trung

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Tất cả config chỉnh tại đây
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# --- Paths ---
_suffix = 'qwen3-1.7b-merge'
base_input_path = './data/'

TRAIN_PATH = f'{base_input_path}train.csv'
VAL_PATH   = f'{base_input_path}val.csv'
TEST_PATH  = f'{base_input_path}test.csv'
GOLD_TEST_PATH = f'{base_input_path}gold_test.csv'
OUTPUT_DIR = f'./outputs-{_suffix}'
CKPT_DIR   = f'./checkpoints-{_suffix}'
BEST_MODEL_PATH = f'{CKPT_DIR}/best'
INFER_PARTIAL_PATH = f'{OUTPUT_DIR}/test_predictions.partial.csv'
INFER_PRED_PATH    = f'{OUTPUT_DIR}/test_predictions.csv'
INFER_GOLD_PRED_PATH = f'{OUTPUT_DIR}/gold_test_predictions.csv'

# --- Generator ---
GENERATOR_MODEL = 'Qwen/Qwen3-1.7B'   # decoder-only causal LM
# GENERATOR_MODEL = 'vinai/bartpho-syllable-base'
# GENERATOR_MODEL = 'VietAI/vit5-large'

# Suffix appended to the prompt; the model generates the answer after it
PROMPT_SUFFIX = '\ntrả lời: '

# --- Retrieval ---
EMBEDDING_MODEL = 'BAAI/bge-m3'
RERANKER_MODEL  = 'BAAI/bge-reranker-v2-m3'
RETRIEVAL_TOP_K = 20
RERANK_TOP_K    = 3

# --- Training ---
MAX_INPUT_LEN    = 1500    # max prompt tokens (truncated from left if needed)
MAX_TARGET_LEN   = 256    # max answer tokens
TRAIN_BATCH_SIZE = 12
EVAL_BATCH_SIZE  = 24
GRAD_ACCUM       = 4      # effective batch = 4 * 4 = 16
NUM_EPOCHS       = 3
LEARNING_RATE    = 2e-5
WARMUP_RATIO     = 0.05
EARLY_STOPPING_PATIENCE  = 2
EARLY_STOPPING_THRESHOLD = 0.0

# --- Inference resilience ---
INFER_SAVE_EVERY = 25

# --- Evaluation ---
BERTSCORE_MODEL = 'vinai/phobert-base'
BERTSCORE_BATCH = 32

# RAGAS + Gemini evaluator config
EVAL_LLM_PROVIDER = 'google'
EVAL_LLM_MODEL = 'gemini-2.5-pro'
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')  # API for thesis 1
GOOGLE_API_KEY = GEMINI_API_KEY or os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY')
USE_RAGAS_METRICS = True

SEED = 42

# --- Input prompt ---
def build_input(question: str, contexts: list) -> str:
    ctx_block = ' '.join([f'[{i+1}] {c}' for i, c in enumerate(contexts)])
    return f'câu hỏi: {question} ngữ cảnh: {ctx_block}'

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
print('Config loaded ✓')
if USE_RAGAS_METRICS:
    if not GOOGLE_API_KEY:
        raise ValueError("Missing Gemini API key. Set GEMINI_API_KEY variable or env GOOGLE_API_KEY/GEMINI_API_KEY.")
    print(f'Eval LLM  : {EVAL_LLM_PROVIDER} | {EVAL_LLM_MODEL}')

## 3. Load & kiểm tra dataset

In [ ]:
def load_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    df['question_type'] = df['question_type'].str.upper().str.strip()
    required = {'question', 'context', 'answer', 'question_type'}
    assert required.issubset(df.columns), f'{path} thiếu cột: {required - set(df.columns)}'
    df = df.dropna(subset=list(required)).reset_index(drop=True)
    return df

train_df = load_csv(TRAIN_PATH)
val_df   = load_csv(VAL_PATH)
test_df  = load_csv(TEST_PATH)
gold_test_df = load_csv(GOLD_TEST_PATH)

print(f'Train     : {len(train_df):>6,}')
print(f'Val       : {len(val_df):>6,}')
print(f'Test      : {len(test_df):>6,}')
print(f'Gold test : {len(gold_test_df):>6,}')
print(f'Total     : {len(train_df)+len(val_df)+len(test_df)+len(gold_test_df):>6,}')

print('\nPhân bổ QA type:')
dist = pd.concat([
    train_df['question_type'].value_counts().rename('train'),
    val_df['question_type'].value_counts().rename('val'),
    test_df['question_type'].value_counts().rename('test'),
    gold_test_df['question_type'].value_counts().rename('gold_test'),
], axis=1).fillna(0).astype(int)
dist['total'] = dist.sum(axis=1)
print(dist.to_string())

print('\nVí dụ 1 mẫu (train):')
for col in ['question_type', 'question', 'context', 'answer']:
    print(f'  {col:10}: {str(train_df.iloc[0][col])[:110]}')

## 4. Xây dựng Vector Store (Offline RAG)

In [ ]:
print('Loading embedding model...')
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

corpus = pd.concat([
    train_df['context'],
    val_df['context'],
    test_df['context'],
    gold_test_df['context'],
]).drop_duplicates().reset_index(drop=True).tolist()
print(f'Corpus: {len(corpus):,} unique chunks')

# Dense index
print('Encoding corpus...')
corpus_embs = embed_model.encode(
    corpus, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True)
faiss_index = faiss.IndexFlatIP(corpus_embs.shape[1])
faiss_index.add(corpus_embs)

# BM25 sparse index
bm25_index = BM25Okapi([doc.lower().split() for doc in corpus])

faiss.write_index(faiss_index, f'{OUTPUT_DIR}/faiss.index')
with open(f'{OUTPUT_DIR}/corpus.json', 'w', encoding='utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False)

print(f'FAISS : {faiss_index.ntotal:,} vectors | dim={corpus_embs.shape[1]}')
print('BM25  : built ✓  |  Index saved ✓')

## 5. Hybrid Retrieval + Cross-Encoder Reranker

In [ ]:
print('Loading reranker...')
reranker = CrossEncoder(RERANKER_MODEL, device=DEVICE)
print('Reranker loaded ✓')

def rrf(lists: list, k: int = 60) -> list:
    scores = defaultdict(float)
    for ranked in lists:
        for rank, doc_id in enumerate(ranked):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

def retrieve(query: str,
             top_k: int = RETRIEVAL_TOP_K,
             rerank_k: int = RERANK_TOP_K) -> list:
    # BM25
    bm25_ids = np.argsort(bm25_index.get_scores(query.lower().split()))[::-1][:top_k].tolist()
    # Dense
    q_emb = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    _, dense_ids = faiss_index.search(q_emb, top_k)
    # RRF + Rerank
    merged     = rrf([bm25_ids, dense_ids[0].tolist()])[:top_k]
    candidates = [corpus[i] for i in merged]
    scores     = reranker.predict([(query, c) for c in candidates])
    top_idx    = np.argsort(scores)[::-1][:rerank_k]
    return [candidates[i] for i in top_idx]

# Kiểm tra nhanh
q0 = train_df.iloc[0]['question']
print(f'\nQuery : {q0[:80]}')
for i, c in enumerate(retrieve(q0)):
    print(f'  [{i+1}] {c[:90]}...')

## 6. Fine-tune Generator


In [ ]:
def load_tokenizer(model_name: str):
    """Load tokenizer for Qwen3-0.6B (decoder-only)."""
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    # Qwen3 has no pad_token by default — set to eos
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # Ensure padding is on the right for causal LM training
    tok.padding_side = 'right'
    return tok

print(f'Loading: {GENERATOR_MODEL}')
tokenizer = load_tokenizer(GENERATOR_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    dtype=torch.bfloat16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE)

model.gradient_checkpointing_enable()
model.config.use_cache = False   # required during training (gradient checkpointing compat)

print(f'Parameters : {sum(p.numel() for p in model.parameters())/1e6:.0f}M')
print(f'Device     : {next(model.parameters()).device}')
print(f'Vocab size : {tokenizer.vocab_size:,}')
print(f'Pad token  : {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})')


In [ ]:
class QADataset(Dataset):
    """
    Causal-LM dataset: full sequence = [prompt + answer + <eos>]
    Labels: -100 for all prompt tokens (masked from loss), answer + <eos> for training signal.
    """
    def __init__(self, dataframe, tokenizer, max_input, max_target):
        self.data      = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_tgt   = max_target

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row    = self.data.iloc[idx]
        prompt = build_input(row['question'], [row['context']]) + PROMPT_SUFFIX
        answer = str(row['answer'])

        # Tokenize separately (no special tokens on answer to avoid double-eos)
        prompt_ids = self.tokenizer(prompt, add_special_tokens=True)['input_ids']
        answer_ids = self.tokenizer(answer, add_special_tokens=False)['input_ids']
        eos_id     = [self.tokenizer.eos_token_id]

        # Truncate: keep the tail of prompt (most recent context) and truncate answer head
        if len(prompt_ids) > self.max_input:
            prompt_ids = prompt_ids[-self.max_input:]
        answer_ids = answer_ids[:self.max_tgt]

        input_ids = prompt_ids + answer_ids + eos_id
        labels    = [-100] * len(prompt_ids) + answer_ids + eos_id

        return {'input_ids': input_ids, 'labels': labels}


train_dataset = QADataset(train_df, tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
val_dataset   = QADataset(val_df,   tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
print(f'Train : {len(train_dataset):,}  |  Val : {len(val_dataset):,}')
# Quick sanity check
ex = train_dataset[0]
n_prompt = sum(1 for l in ex['labels'] if l == -100)
n_answer = sum(1 for l in ex['labels'] if l != -100)
print(f'Sample 0 — total tokens: {len(ex["input_ids"])} | prompt (masked): {n_prompt} | answer: {n_answer}')


In [ ]:
from torch.nn.utils.rnn import pad_sequence

class CausalLMCollator:
    """Right-pad input_ids and labels; create attention_mask."""
    def __init__(self, pad_token_id: int):
        self.pad_id = pad_token_id

    def __call__(self, features):
        input_ids = [torch.tensor(f['input_ids'], dtype=torch.long) for f in features]
        labels    = [torch.tensor(f['labels'],    dtype=torch.long) for f in features]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.pad_id)
        labels    = pad_sequence(labels,    batch_first=True, padding_value=-100)
        attention_mask = (input_ids != self.pad_id).long()

        return {'input_ids': input_ids, 'labels': labels, 'attention_mask': attention_mask}


data_collator = CausalLMCollator(tokenizer.pad_token_id)

training_args = TrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = TRAIN_BATCH_SIZE,
    per_device_eval_batch_size  = EVAL_BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    optim                       = "paged_adamw_8bit",

    warmup_steps = int(
        WARMUP_RATIO *
        (len(train_dataset) // (TRAIN_BATCH_SIZE * GRAD_ACCUM)) *
        NUM_EPOCHS
    ),

    weight_decay                = 0.01,
    tf32                        = False,
    bf16                        = True,
    torch_compile               = True,

    # CHANGED ↓
    eval_strategy               = "steps",
    eval_steps                  = 400,

    save_strategy               = "steps",
    save_steps                  = 400,

    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,

    logging_steps               = 50,
    save_total_limit            = 2,
    report_to                   = "none",
    seed                        = SEED,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    processing_class = tokenizer,
    data_collator   = data_collator,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)]
)
print(f'Effective batch size : {TRAIN_BATCH_SIZE * GRAD_ACCUM}')
print(f'Steps / epoch        : ~{len(train_dataset) // (TRAIN_BATCH_SIZE * GRAD_ACCUM)}')


In [ ]:
train_result = trainer.train()
best_path = f'{CKPT_DIR}/best'
trainer.save_model(best_path)
tokenizer.save_pretrained(best_path)
print(f'Training complete ✓  |  Train loss: {train_result.training_loss:.4f}')
print(f'Best model → {best_path}')

In [ ]:
log_history = trainer.state.log_history
train_logs  = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_logs   = [x for x in log_history if 'eval_loss' in x]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if train_logs:
    axes[0].plot([x['step'] for x in train_logs],
                 [x['loss'] for x in train_logs], color='steelblue', lw=1.5)
    axes[0].set(title='Training Loss', xlabel='Step', ylabel='Loss')
    axes[0].grid(alpha=0.3)
if eval_logs:
    axes[1].plot([x['epoch'] for x in eval_logs],
                 [x['eval_loss'] for x in eval_logs], color='coral', marker='o', lw=1.5)
    axes[1].set(title='Validation Loss', xlabel='Epoch', ylabel='Loss')
    axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=600, bbox_inches='tight')
plt.show()

## 7. Inference — Sinh câu trả lời trên Test Set

In [ ]:
print(f'Loading best model from {best_path}...')
tokenizer = AutoTokenizer.from_pretrained(best_path, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'   # left-pad for batch inference with causal LM

model = AutoModelForCausalLM.from_pretrained(
    best_path,
    dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE)
model.config.use_cache = True   # re-enable KV-cache for faster inference
model.eval()
print('Model loaded ✓')


In [ ]:
# Inference — Causal LM (Qwen3-0.6B)
# Key difference vs Seq2Seq: we must slice off the prompt tokens from the output
# before decoding, otherwise the prediction contains the full input + answer.

def _make_gen_config(question_type: str = '') -> GenerationConfig:
    no_repeat = 0 if question_type == 'VERIFICATION' else 2
    return GenerationConfig(
        max_new_tokens       = MAX_TARGET_LEN,
        num_beams            = 4,
        no_repeat_ngram_size = no_repeat,
        early_stopping       = True,
        pad_token_id         = tokenizer.pad_token_id,
        eos_token_id         = tokenizer.eos_token_id,
    )


@torch.inference_mode()
def generate_answer(question: str, chunks: list, question_type: str = '') -> tuple:
    top_chunk = chunks[0]             # best context after reranking
    prompt    = build_input(question, [top_chunk]) + PROMPT_SUFFIX
    inputs    = tokenizer(
        prompt, return_tensors='pt',
        max_length=MAX_INPUT_LEN, truncation=True,
    ).to(DEVICE)
    prompt_len = inputs['input_ids'].shape[1]   # number of prompt tokens

    gc      = _make_gen_config(question_type)
    ids     = model.generate(**inputs, generation_config=gc)

    # Decode only the newly generated tokens (skip the prompt prefix)
    new_ids = ids[0][prompt_len:]
    decoded = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    answer  = decoded if decoded else '[NO_ANSWER]'
    return answer, top_chunk


def run_inference(split_df: pd.DataFrame, split_name: str, output_path: str) -> pd.DataFrame:
    print(f'Running inference on {len(split_df):,} {split_name} samples...')
    predictions, contexts_used = [], []
    for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
        chunks = retrieve(row['question'])
        ans, ctx = generate_answer(row['question'], chunks, row['question_type'])
        predictions.append(ans)
        contexts_used.append(ctx)

    results = split_df.copy()
    results['prediction'] = predictions
    results['context_used'] = contexts_used
    results.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f'Inference complete ✓  |  Saved → {output_path}')
    print('\nSample predictions:')
    for _, row in results.head(3).iterrows():
        print(f'  [{row["question_type"]}] Q: {str(row["question"])[:60]}')
        print(f'         Pred : {str(row["prediction"])[:80]}')
        print(f'         Gold : {str(row["answer"])[:80]}')
        print()

    return results


test_results = run_inference(test_df, 'test', INFER_PRED_PATH)
gold_test_results = run_inference(gold_test_df, 'gold test', INFER_GOLD_PRED_PATH)

# test_results = pd.read_csv(INFER_PRED_PATH)
# gold_test_results = pd.read_csv(INFER_GOLD_PRED_PATH)

print(test_results.shape)
print(gold_test_results.shape)